# Social Mobility — A Measurement of a Place That No Longer Exists

The ACS cannot answer this domain's question. Mobility is intergenerational, and a
cross-sectional survey has no link between a child and the household they grew up in. The
Opportunity Atlas does have that link: it follows children from federal tax and Census records
into adulthood.

What it gives back is stranger than it first appears. A county's mobility value describes
**children born 1978–1983, located by where they grew up, with adult outcomes measured at
age 35.** Those adults are now in their forties. The number is a statement about a place in
the 1980s and 1990s, and it does not update when the place changes. Two counties with
identical values today may be nothing alike today.

This notebook takes that seriously, and then asks what the estimates will and will not support.

## Learning Objectives

1. **Separate a release date from a reference period** when the reference period is a cohort's
   childhood rather than a year.
2. **Measure who disclosure protection removes**, and notice when the removal is patterned
   rather than random.
3. **Read a count that cannot be a count** as evidence of what was done to the data.
4. **Settle whether rank failure is about precision** by testing it where precision is good.

## Prerequisites

- Handbook §7.1 and §7.2 — this notebook supplies the decisive case for both.
- D04 for the contrast: PLACES is *modelled*, this is *measured*, and they fail the same test.

## Data Sources

| file | geography | what a row is |
|---|---|---|
| `county_outcomes_simple.csv` | county | children who **grew up** in that county, born 1978–1983 |

From Opportunity Insights. Fetched at runtime rather than committed: the corpus has no
verified redistribution licence for these files, and the same caution that keeps Zillow's
downloads out of the repository applies. A test enforces that they stay uncommitted.

Opportunity Insights' own FAQ supplies the three facts this notebook rests on — the 1978–1983
cohort measured at age 35, the mapping by where children grew up, and that random noise is
added to all estimates with suppression below 20 children.

In [1]:
"""Configuration."""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path.cwd().resolve()
for _cand in (_here, *_here.parents):
    if (_cand / "src" / "krl_tutorials_data").is_dir():
        sys.path.insert(0, str(_cand / "src"))
        break
else:
    raise SystemExit("Could not locate krl_tutorials_data from " + str(_here))

from krl_tutorials_data.sources import (
    OPPORTUNITY_ATLAS_COHORT, fetch_acs, fetch_opportunity_atlas,
)

STATE_FIPS = "51"
ACS_YEAR = 2023
Z95 = 1.959964
#: The repository root, found by walking UP from wherever this runs. Provenance
#: used to be written to `Path.cwd() / "provenance"` in some notebooks and to a
#: repo-root-relative path in others, so the two conventions could not both be
#: satisfied by one working directory -- and a run from the wrong one silently
#: wrote sixteen records into a stray folder at the top of the repo.
REPO_ROOT = next(c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                 if (c / "src" / "krl_tutorials_data").is_dir())
OUT = REPO_ROOT / "projects" / "05-social-systems/D10_social_mobility" / "provenance"
# kfr = "kid family rank": the child's mean income rank in adulthood, for children
# whose parents sat at the 25th percentile of the national income distribution.
OUTCOME = "kfr_pooled_pooled_p25"
SUBGROUPS = {
    "kfr_pooled_pooled_p25": "all children",
    "kfr_white_pooled_p25": "White",
    "kfr_black_pooled_p25": "Black",
    "kfr_hisp_pooled_p25": "Hispanic",
    "kfr_asian_pooled_p25": "Asian",
    "kfr_natam_pooled_p25": "American Indian / Alaska Native",
}
print(f"Opportunity Atlas cohort: {OPPORTUNITY_ATLAS_COHORT}")

Opportunity Atlas cohort: children born 1978-1983, outcomes at age 35


In [2]:
"""Retrieve the Atlas, and today's ACS for the same counties."""
atlas, prov_atlas = fetch_opportunity_atlas(geography="county", state_fips=STATE_FIPS)
written = [prov_atlas.write(OUT / "d10_opportunity_atlas_county.json")]
print(f"{len(atlas)} Virginia counties and independent cities")
print()
print(prov_atlas.render()[:640])

133 Virginia counties and independent cities

Source      : Opportunity Insights (Opportunity Atlas) - Opportunity Atlas outcomes, county level
Vintage     : children born 1978-1983, outcomes at age 35  (pinned)
Endpoint    : https://opportunityinsights.org/wp-content/uploads/2018/10/county_outcomes_simple.csv
Retrieved   : 2026-09-18T12:13:01+00:00
Shape       : 133 rows x 65 cols
Payload hash: c355f7cade477637...  (response-bytes)
Data mode   : EMPIRICAL
Variables   :
              kfr_pooled_pooled_p25 see Opportunity Insights codebook
              kfr_pooled_pooled_p25_se see Opportunity Insights codebook
              jail_pooled_pooled_p25 see Opportunity Insights codebo


## 1. The vintage is a cohort, and that is not a technicality

Every other source in this corpus has a vintage that is a date: a survey year, a release, a
quarter. Here the provenance records a **cohort**, because that is what the reference period
is. Recording "2018" — the year the file was published — would be the central misreading of
this source.

The gap matters because places change. Compare the Atlas against the ACS for the same counties
today, and the question is not whether they agree. It is what each one is about.

In [3]:
"""How far apart are the Atlas cohort and the present-day population of the same counties?"""
acs, prov_acs = fetch_acs(
    year=ACS_YEAR,
    variables={"B01003_001E": "total population", "B01003_001M": "total population MOE",
               "B19013_001E": "median household income", "B19013_001M": "median income MOE"},
    geography="county:*", within=f"state:{STATE_FIPS}",
)
acs["fips"] = acs["state"] + acs["county"]
written.append(prov_acs.write(OUT / "d10_acs_county_today.json"))

merged = atlas.merge(acs, on="fips", how="inner")
print(f"counties matched on FIPS: {len(merged)} of {len(atlas)}")
print()
print("The Atlas ships `state` and `county` as UNPADDED integers -- Accomack is")
print("state=51, county=1. Concatenating without zero-padding gives '511', which")
print("matches nothing and produces an EMPTY join rather than an error. The fetcher")
print("builds a padded `fips` column so this cannot happen silently.")
print()
birth_first, birth_last, measured_age = 1978, 1983, 35
outcome_last = birth_last + measured_age
print(f"Born                 {birth_first}-{birth_last}")
print(f"Childhood in place   ~{birth_first}-{birth_last + 18}")
print(f"Outcomes measured    ~{birth_first + measured_age}-{outcome_last}  (at age {measured_age})")
print(f"File published       2018")
print(f"ACS beside it        {ACS_YEAR - 4}-{ACS_YEAR}")
print()
print(f"Note the trap: the last outcomes were observed around {outcome_last}, and the file")
print(f"was published in 2018. The two coincide, which makes '2018' look like a")
print("defensible vintage. It is not -- it is the year the LAST cohort member turned 35.")
print()
print(f"What the county value characterises is the place during the cohort's CHILDHOOD,")
print(f"roughly {birth_first}-{birth_last + 18}. For a reader in {ACS_YEAR} that is between")
print(f"{ACS_YEAR - (birth_last + 18)} and {ACS_YEAR - birth_first} years ago, and the Atlas population is defined by")
print("where it was then, not where it is now.")

counties matched on FIPS: 133 of 133

The Atlas ships `state` and `county` as UNPADDED integers -- Accomack is
state=51, county=1. Concatenating without zero-padding gives '511', which
matches nothing and produces an EMPTY join rather than an error. The fetcher
builds a padded `fips` column so this cannot happen silently.

Born                 1978-1983
Childhood in place   ~1978-2001
Outcomes measured    ~2013-2018  (at age 35)
File published       2018
ACS beside it        2019-2023

Note the trap: the last outcomes were observed around 2018, and the file
was published in 2018. The two coincide, which makes '2018' look like a
defensible vintage. It is not -- it is the year the LAST cohort member turned 35.

What the county value characterises is the place during the cohort's CHILDHOOD,
roughly 1978-2001. For a reader in 2023 that is between
22 and 45 years ago, and the Atlas population is defined by
where it was then, not where it is now.


A county that gentrified, deindustrialised, or absorbed a new commuter suburb since 1990 has
an Atlas value describing none of that. The measurement is sound; the thing it measures is
simply not "opportunity in this county now".

## 2. Who the disclosure rule removes

Opportunity Insights suppresses cells with fewer than 20 children — in a tract, or of a given
race and gender in a county. That is a sound privacy rule and it is applied evenly.

Applied evenly to uneven populations, it does not remove people evenly.

In [4]:
"""Suppression by subgroup."""
rows = []
for col, label in SUBGROUPS.items():
    if col not in atlas.columns:
        continue
    missing = int(atlas[col].isna().sum())
    rows.append({
        "subgroup": label, "column": col,
        "counties with an estimate": len(atlas) - missing,
        "suppressed": missing,
        "share suppressed": f"{100 * missing / len(atlas):5.1f}%",
    })
suppression = pd.DataFrame(rows)
print(suppression.to_string(index=False))
print()
print("The pooled estimate is available everywhere. The subgroup estimates are not, and")
print("what is missing is ordered by population size -- which in Virginia means the")
print("smallest groups lose the most counties.")

    subgroup                column  counties with an estimate  suppressed share suppressed
all children kfr_pooled_pooled_p25                        133           0             0.0%
       White  kfr_white_pooled_p25                        133           0             0.0%
       Black  kfr_black_pooled_p25                        119          14            10.5%
    Hispanic   kfr_hisp_pooled_p25                         90          43            32.3%

The pooled estimate is available everywhere. The subgroup estimates are not, and
what is missing is ordered by population size -- which in Virginia means the
smallest groups lose the most counties.


This is the part worth sitting with. A researcher asking "does this county produce different
outcomes for Black and White children who grew up here?" needs both subgroup columns in the
same county. Wherever either is suppressed, the comparison is unavailable — and suppression
concentrates precisely where the minority population is small, which is a large share of
Virginia's counties.

The rule is not discriminatory; it is a count threshold applied identically. But its *effect*
is that the equity question is answerable in the places where it is least likely to be asked,
and unanswerable in many of the places where a disparity would matter most to identify. That
is a property of the published data, not a criticism of the privacy standard, and an analysis
that quietly drops suppressed rows inherits it without saying so.

In [5]:
"""How many counties support a Black/White comparison at all?"""
pair = atlas[["kfr_white_pooled_p25", "kfr_black_pooled_p25"]]
both = int(pair.notna().all(axis=1).sum())
white_only = int((pair.kfr_white_pooled_p25.notna() & pair.kfr_black_pooled_p25.isna()).sum())

print(f"counties with BOTH estimates      {both:>4} of {len(atlas)}   ({100 * both / len(atlas):.1f}%)")
print(f"counties with White but not Black {white_only:>4}")
print()
gap = (atlas.kfr_white_pooled_p25 - atlas.kfr_black_pooled_p25).dropna()
print(f"Where both exist, the White-Black gap in adult income rank averages "
      f"{gap.mean():+.4f}")
print(f"  (rank units, 0-1). Range {gap.min():+.4f} to {gap.max():+.4f}.")
print()
print("That average is computed over a NON-RANDOM subset of counties -- the ones with")
print("enough Black children in the 1978-1983 cohort to clear the threshold. It is not")
print("the state's gap, and this data cannot produce the state's gap.")

counties with BOTH estimates       119 of 133   (89.5%)
counties with White but not Black   14

Where both exist, the White-Black gap in adult income rank averages +0.0896
  (rank units, 0-1). Range -0.0023 to +0.1659.

That average is computed over a NON-RANDOM subset of counties -- the ones with
enough Black children in the 1978-1983 cohort to clear the threshold. It is not
the state's gap, and this data cannot produce the state's gap.


## 3. A count that cannot be a count

Opportunity Insights adds random noise to every estimate. The clearest evidence is not in the
estimates, where it is invisible, but in the counts — which are published with fractional
parts. People do not come in fractions.

In [6]:
"""Check the counts."""
counts = pd.to_numeric(atlas["pooled_pooled_count"], errors="coerce").dropna()
fractional = int(((counts % 1).abs() > 1e-9).sum())

print(f"counties with a count          {len(counts)}")
print(f"counts with a fractional part  {fractional}   ({100 * fractional / len(counts):.1f}%)")
print(f"examples: {[round(v, 4) for v in counts.head(4)]}")
print()
print("Every single one. A count of children is an integer by construction, so a column")
print("where none of them are integers is not a rounding artifact -- it is the noise")
print("made visible in the one place it cannot hide.")
print()
print("The consequence for analysis: these counts cannot be used as weights or")
print("denominators without acknowledging that they are perturbed, and small counts are")
print("proportionally more perturbed than large ones.")

counties with a count          133
counts with a fractional part  133   (100.0%)
examples: [5855.1777, 5689.1904, 1707.3881, 1486.6022]

Every single one. A count of children is an integer by construction, so a column
where none of them are integers is not a rounding artifact -- it is the noise
made visible in the one place it cannot hide.

The consequence for analysis: these counts cannot be used as weights or
denominators without acknowledging that they are perturbed, and small counts are
proportionally more perturbed than large ones.


## 4. The decisive test for handbook §7.2

§7.1 found no determined rank among 13,981 tracts. §7.2 found none among 2,956 counties on
CDC PLACES — but PLACES is *modelled*, and a fair objection is that the failure belongs to the
model rather than to ranking.

This source answers that objection. It is **measured**, from administrative records rather
than a survey or a model. And its signal-to-noise is the best in the corpus: for the first
time, the interval around a county is **narrower** than the spread across counties.

In [7]:
"""Reliability and rank identifiability on a measured, well-powered estimate."""
est_s = atlas[OUTCOME]
se_s = atlas[f"{OUTCOME}_se"]
ok = est_s.notna() & se_s.notna() & (se_s > 0)
est, se = est_s[ok].to_numpy(float), se_s[ok].to_numpy(float)
lo, hi = est - Z95 * se, est + Z95 * se

between_sd = float(est.std(ddof=1))
ci_width = float(np.median(hi - lo))
tau2 = max(float(est.var(ddof=1) - (se ** 2).mean()), 0.0)
lam = tau2 / (tau2 + se ** 2)

determined = sum(
    1 for i in range(len(est))
    if not ((lo < hi[i]) & (hi > lo[i]) & (np.arange(len(est)) != i)).any()
)
best = np.array([1 + int((lo > hi[i]).sum()) for i in range(len(est))])
worst = np.array([len(est) - int((hi < lo[i]).sum()) for i in range(len(est))])

print(f"counties with estimate and SE     {len(est)}")
print(f"between-county SD                 {between_sd:.4f}")
print(f"median 95% interval width         {ci_width:.4f}")
print(f"ratio (width / SD)                {ci_width / between_sd:.2f}   <- below 1 for the")
print(f"                                          first time in this corpus")
print(f"pooled reliability lambda         {tau2 / (tau2 + float((se ** 2).mean())):.3f}")
print(f"share of counties lambda >= 0.7    {100 * (lam >= 0.7).mean():.0f}%")
print()
print(f"determined ranks                  {determined} of {len(est)}")
print(f"median rank span (deterministic)  {np.median(worst - best):.0f} of {len(est)}")

counties with estimate and SE     133
between-county SD                 0.0372
median 95% interval width         0.0333
ratio (width / SD)                0.89   <- below 1 for the
                                          first time in this corpus
pooled reliability lambda         0.916
share of counties lambda >= 0.7    98%

determined ranks                  0 of 133
median rank span (deterministic)  80 of 133


**Measured, not modelled. Reliability 0.916. The interval narrower than the spread. Zero
determined ranks.**

That settles what §7.2 could only argue. Rank failure is not a property of modelling, and it
is not a symptom of weak precision. Telling one unit from 133 requires every pairwise
comparison to separate simultaneously, and the number of comparisons grows with the square of
the list. No achievable precision fixes it, because it is not a precision problem.

The estimands §7.2 recommends — detection against a benchmark, tiers, exceedance — apply here
unchanged, and this notebook's §5 uses the one that suits the question.

In [8]:
"""What the data support: separation from the state benchmark, not an order."""
w = 1 / se ** 2
benchmark = float((w * est).sum() / w.sum())
z = (est - benchmark) / se
outside95 = float((np.abs(z) > Z95).mean())
outside998 = float((np.abs(z) > 3.090232).mean())

print(f"precision-weighted state benchmark   {benchmark:.4f}")
print(f"counties outside the 95% limits      {100 * outside95:5.1f}%")
print(f"counties outside the 99.8% limits     {100 * outside998:5.1f}%")
print()
print(f"Ranking named {determined} counties. This names a definite set, and every county")
print("in it is separated from the state as a whole rather than from a neighbour.")
print()
print("For this domain that is also the better question. 'Which county is 7th best for")
print("mobility' has no policy action attached to it. 'Which counties produced outcomes")
print("clearly below the state benchmark for children who grew up there in the 1980s'")
print("does -- as a starting point for asking what was true of those places then.")

precision-weighted state benchmark   0.3900
counties outside the 95% limits       56.4%
counties outside the 99.8% limits      42.1%

Ranking named 0 counties. This names a definite set, and every county
in it is separated from the state as a whole rather than from a neighbour.

For this domain that is also the better question. 'Which county is 7th best for
mobility' has no policy action attached to it. 'Which counties produced outcomes
clearly below the state benchmark for children who grew up there in the 1980s'
does -- as a starting point for asking what was true of those places then.


In [9]:
"""FINDINGS"""
print("FINDINGS")
print("=" * 78)
print(f"""
1. The vintage is a COHORT, not a date: children born 1978-1983, measured at
   age 35, located by where they GREW UP. The last outcomes were observed
   around {1983 + 35} -- the same year the file was published, which is a
   coincidence that makes "2018" look like a defensible vintage. It is not.
   What a county value characterises is that place during the cohort's
   CHILDHOOD, roughly 1978-2001, which for a reader today is a generation ago.
   The number does not update when the county changes.

2. Disclosure suppression is applied evenly to uneven populations and therefore
   removes unevenly. In Virginia the pooled estimate is present for all
   {len(atlas)} counties; the Black estimate is missing for
   {int(atlas.kfr_black_pooled_p25.isna().sum())}, the Hispanic estimate for
   {int(atlas.kfr_hisp_pooled_p25.isna().sum())}. Only {both} counties support a
   Black/White comparison at all, so any statewide gap computed from this
   source is computed over a non-random subset and is not the state's gap.

3. Every published count carries a fractional part ({fractional} of {len(counts)}).
   A count of children cannot be fractional, so the column is the injected
   privacy noise made visible. Small counts are proportionally more perturbed.

4. This source settles handbook §7.2's open objection. It is MEASURED rather
   than modelled, its reliability is {tau2 / (tau2 + float((se ** 2).mean())):.3f} with {100 * (lam >= 0.7).mean():.0f}% of counties
   clearing 0.7, and its interval is NARROWER than the between-county spread
   (ratio {ci_width / between_sd:.2f}, the first time in this corpus) -- and it still yields
   {determined} determined ranks out of {len(est)}. Rank failure is neither a modelling
   artifact nor a precision problem.

5. What the data do support: {100 * outside95:.1f}% of counties separate from the
   precision-weighted state benchmark at 95%, {100 * outside998:.1f}% at 99.8%. That is a
   nameable set where ranking produced none.
""")
print(f"Provenance written: {len(written)} records")
for p in written:
    print(f"  {p}")

FINDINGS

1. The vintage is a COHORT, not a date: children born 1978-1983, measured at
   age 35, located by where they GREW UP. The last outcomes were observed
   around 2018 -- the same year the file was published, which is a
   coincidence that makes "2018" look like a defensible vintage. It is not.
   What a county value characterises is that place during the cohort's
   CHILDHOOD, roughly 1978-2001, which for a reader today is a generation ago.
   The number does not update when the county changes.

2. Disclosure suppression is applied evenly to uneven populations and therefore
   removes unevenly. In Virginia the pooled estimate is present for all
   133 counties; the Black estimate is missing for
   14, the Hispanic estimate for
   43. Only 119 counties support a
   Black/White comparison at all, so any statewide gap computed from this
   source is computed over a non-random subset and is not the state's gap.

3. Every published count carries a fractional part (133 of 133).
   A

## Key Takeaways

1. **A release year is not a reference period.** Here the reference period is a childhood that
   ended decades before publication, and the distinction changes every sentence written about
   the number.
2. **An even rule applied to uneven populations removes unevenly.** Check who a suppression
   threshold takes out before analysing what remains.
3. **Impossible values are evidence.** A fractional count tells you what was done to the data.
4. **Rank failure is structural.** Demonstrated here on a measured estimate with favourable
   precision, which rules out the two obvious explanations.

## Exercises

1. **Date the mismatch per county.** Using ACS population change since 1990 (or the Atlas's
   own `tract_covariates.csv`), find the Virginia counties that changed most since the cohort's
   childhood. Those are where reading the Atlas as a statement about today goes most wrong.
2. **Quantify the suppression bias.** Compare the counties that DO support a Black/White
   comparison against those that do not, on population and median income. How is the
   comparable subset unrepresentative, and in which direction?
3. **Test the noise.** The counts are perturbed. Estimate the perturbation's scale by examining
   the distribution of fractional parts, and state whether it looks uniform.
4. **Reproduce §4 at tract level.** Suppression is much harsher there. Predict before running
   whether the determined-rank count stays at zero, and what happens to reliability.
5. **Find the honest headline.** Write one sentence about Virginia mobility that this source
   supports, and one that it does not but that a newspaper might print. Explain the difference.

## References

- Opportunity Insights, Frequently Asked Questions — source of the 1978–1983 cohort, the
  age-35 measurement, the "mapped by where children grew up" rule, the noise injection and the
  20-child suppression threshold. *Retrieved and verified during this build.*
- Chetty, Friedman, Hendren, Jones & Porter, *The Opportunity Atlas: Mapping the Childhood
  Roots of Social Mobility.* *(Citation not verified in this build — treat as a lead.)*
- Handbook §1.9 (the linked-longitudinal regime), §7.1 and §7.2 (rankings).
- D04 for the modelled-estimate contrast.

## Limitations

- **One state, one cohort.** Virginia's 133 counties, children born 1978–1983. There is no
  later cohort at this geography in this file, so nothing here speaks to whether mobility has
  changed.
- **The suppression bias is measured for its extent, not corrected.** §2 reports how many
  counties are lost and notes the subset is non-random; it does not reweight, and exercise 2
  asks only for a characterisation rather than a fix.
- **The noise is acknowledged, not modelled.** The published standard errors do not obviously
  incorporate the injected noise, and this build did not establish whether they do. Every
  interval in §4 inherits that uncertainty about its own uncertainty.
- **`kfr` is a rank, not a dollar amount.** It is the child's mean position in the national
  income distribution, so a county can improve its rank only at another's expense. Nothing
  here measures absolute gains.
- **Causality is entirely absent.** These are outcomes of children who grew up somewhere, not
  effects of growing up there. Families who moved differ from families who stayed, and this
  notebook does none of the work that would separate place from selection.
- **Not tested here:** whether the §4 result holds at tract level, and whether the published
  SEs are comparable across subgroups with very different cell counts.